# 18 · 第五个工具：查疾病介绍（RAG）

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → [18 疾病介绍（RAG） ✅] → 19 基因工具 → 20 大考
```

> **这一章的代码在合并请求 #20 里。** 建索引在本机做，不花钱；调模型的只有一次 20 道题的试跑（约 ¥0.5），记在 MLflow 里。下面每一格都只读记录、在本机计算，**运行这本笔记不花钱**（第一次运行要加载索引，约半分钟）。

## 背景：公司里最常见的那种 RAG

RAG（retrieval-augmented generation）就是"先查资料，再回答"。我们前面的工具其实都是检索，只是查的是表格（HPO 的注释）和病人（第 17 章）。

公司里说"做 RAG"，通常指另一种：资料是**一大堆文字**（产品文档、合同、病历、指南），先把它们切成小段、转成向量存进向量库，提问时搜出最相关的几段，交给模型照着回答，并标出处。这是做大模型应用最常见的模式，招聘要求里几乎都会写。

对我们有两个用处：
1. **给判断多一份证据**：第 16 章的 ③ 类（看到了、细查了，却没排第一）需要更多关于候选病的信息。疾病介绍里写的东西，HPO 的编号不一定有：发病年龄、病程、哪些表现最典型
2. **回答带出处**：每个诊断都能说"根据 Orphanet 的介绍……"，医生能回头查

## 这一章做的 6 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 准备资料：两份公开的疾病介绍 | Orphanet、MedlinePlus，登记校验码 | 资料来源合法、可复现 |
| 2 | 切块 | 一个定义一块；一段一块，前面加病名 | 搜出来的每一块都能单独看懂 |
| 3 | 向量化，存进向量库 | FastEmbed（本机）+ Qdrant，混合检索 | 按意思找，也按关键词找 |
| 4 | 防泄露 | 遮住"只从本篇论文知道"的病 | 文字不注明出处，没法按论文遮 |
| 5 | 单独评测检索 | 三种检索方式比召回率；不用模型的基线 | 检索不准，模型再聪明也没用 |
| 6 | 接进 agent 和 MCP，能拿掉 | `--without search_disease_texts` | 为第 20 章的消融实验做准备 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 准备资料

### 在干什么
下载两份公开的疾病介绍，登记进数据清单（第 05 章），每个文件都有校验码。

### 背景
RAG 的效果首先取决于资料。我们要的是：覆盖罕见病、能对上 OMIM 编号（其他工具和判分都用它）、可以合法使用。

### 用了什么技术，为什么

| 资料 | 内容 | 许可 |
|---|---|---|
| **Orphanet**（欧洲罕见病数据库） | 每个罕见病一段定义，约 50 个词 | CC BY 4.0：注明出处即可使用 |
| **MedlinePlus Genetics**（美国国家医学图书馆） | 每种遗传病好几段介绍 | 公有领域，注明出处 |

两家都不给下载地址标版本：同一个地址，过几个月下载到的是新文件。所以清单里记下这次用的版本（Orphanet 文件里写的发布日期、MedlinePlus 的下载日期）和校验码；以后上游一变，校验就通不过，不会悄悄换掉。清单里用 `latest_only = true` 写明这一点。

### 公司里什么情况怎么做
- **先查许可证再用数据**：OMIM 的文字质量最高，但商用要授权，所以不用
- **资料要能复现**：记下版本和校验码。上游会变，结果要能对得上当时用的那份

### 执行

In [ ]:
from raredx.data import sources

registry = sources.load_sources(ROOT / "evals/datasets.toml")
for name in ("orphanet", "medlineplus"):
    s = registry[name]
    print(f"{name:<12} 版本 {s.version}   许可 {s.license}")
    print(f"{'':<12} 文件 {', '.join(s.files)}   latest_only={s.latest_only}")

### 结果怎么看
- 两份资料都登记了版本、许可和校验码，`raredx data fetch orphanet medlineplus` 下载时自动核对

### 面试怎么说
> "The texts are Orphanet's definitions (CC BY 4.0) and MedlinePlus Genetics' summaries (public domain), registered with checksums. Neither upstream versions its URL, so the checksum is the pin: a changed upstream fails loudly instead of slipping in."

---

## 第 2 步 · 切块

### 在干什么
把两份资料切成一块一块（passage），每块记下它讲的是哪个（或哪几个）OMIM 病、出自哪里。

### 背景
向量检索是一块一块地比。块太大，一块里讲好几件事，搜得不准；块太小，单独看不懂。

### 用了什么技术，为什么
- **Orphanet**：一个病的定义正好一块（约 50 个词）。只要 Orphanet 写明"和 OMIM 完全对应"的病，避免把一类病的介绍算到某一个亚型头上
- **MedlinePlus**：按段落切，一段一块；每块前面加上病名。不加的话，"It runs in families" 这种句子单独搜出来，不知道在说哪个病
- 只留对得上 OMIM 编号、而且 HPO 里有这个病的块

一共 7,790 块：Orphanet 3,042 块，MedlinePlus 4,748 块。

### 公司里什么情况怎么做
- **切块方式跟着文档结构走**：有天然的段落、条目，就按它切，比"每 500 个字一刀"好
- **每块带上下文**：标题、病名、章节名放进块里，单独搜出来也看得懂

### 执行
第一次运行要加载索引（约 1 万块的向量），大约半分钟。

In [ ]:
from raredx.cli import _text_search

search = _text_search(ROOT / "evals", ROOT / "data")        # 读取第 3 步建好的索引
passages = search.index.passages
orpha = [p for p in passages if p.source.startswith("Orphanet")]
print(f"一共 {len(passages)} 块：Orphanet {len(orpha)}，MedlinePlus {len(passages) - len(orpha)}\n")
for p in (orpha[0], next(p for p in passages if not p.source.startswith("Orphanet"))):
    print(f"[{p.source}]  讲的是 {', '.join(p.disease_ids)}")
    print(f"  {p.text[:300]}\n")

### 结果怎么看
- 每块开头都是病名，后面是介绍
- 每块记着它讲的 OMIM 病和出处：搜出来以后，agent 可以接着用别的工具查这个病，回答里可以写出处

### 面试怎么说
> "Chunking follows the documents: one Orphanet definition per passage, one MedlinePlus paragraph per passage, each led by the disease's name so it reads on its own, and each carrying its OMIM ids and source."

---

## 第 3 步 · 向量化，存进向量库

### 在干什么
给每一块算两个向量，存进向量库 Qdrant；查询时两种方式一起搜，再把结果合并。

### 背景
- **按意思找**（稠密向量）：病人写的是"听不见"，介绍里写的是"hearing loss"，字不一样，意思一样
- **按关键词找**（BM25，稀疏向量）：罕见病里有很多专有名词，比如"hypogammaglobulinemia"，字对上了就很说明问题，按意思找反而容易被冲淡

两种各有长处，公司里通常两种一起用，叫**混合检索**（hybrid search）。

### 用了什么技术，为什么
- **FastEmbed**（Qdrant 出的，Apache-2.0）：在本机算向量，数据不出电脑，也不花钱
- **稠密模型 BAAI/bge-small-en-v1.5**（MIT）：常用的英文检索模型，小而快
- **Qdrant**（Apache-2.0）：常用的向量库。两种向量存在同一个集合里，查询时各搜一份，用"倒数排名融合"（RRF）合并：两份名单里都靠前的排最前
- **算一次，存下来**：给 7,790 块算向量要几分钟，用 `raredx data index` 算一次存进 data/processed。工具启动时直接读进内存里的 Qdrant。换成 Qdrant 服务器，代码只改一个地址

### 公司里什么情况怎么做
- **混合检索是默认选项**：只用向量检索，专有名词、编号、型号常常搜不准
- **向量算一次，别每次重算**：资料不变，向量就不变

### 执行
用一个病人的几个症状当查询，看搜出来什么。

In [ ]:
import mlflow
from raredx.config import settings

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("playground")
print("索引：", search.index.info["sources"], "|", search.index.info["dense"], "+", search.index.info["sparse"], "\n")
query = "Hypogammaglobulinemia; decreased B cell count; recurrent infections"
for hit in search(query, top_k=3).hits:
    print(f"[{hit.source}]  {', '.join(hit.disease_ids)}")
    print(f"  {hit.text[:220]}\n")

### 结果怎么看
- 搜出来的都是抗体缺乏类的免疫缺陷病，每条都有出处和 OMIM 编号
- agent 拿到以后，可以用 `check_match` 逐条核对，或者在理由里写"Orphanet 的介绍说……"

### 面试怎么说
> "Hybrid retrieval: a dense model for meaning and BM25 for exact terms, both computed locally with FastEmbed and fused by reciprocal rank inside Qdrant. Embedding is done once and stored; a tool loads the vectors into an in-memory collection."

---

## 第 4 步 · 防泄露

### 在干什么
考试时，遮住"只从这道题自己的论文才知道"的病的介绍。

### 背景
第 10 章的遮挡是按论文来的：HPO 的每条注释都写着出处，出处是这道题的论文，就遮住。可是疾病介绍**不写出处**，没法按论文遮。

想一想泄露从哪来：第 16 章的 ① 类，这个病只在这道题自己的论文里描述过。那么不管是 Orphanet 还是 MedlinePlus，它们对这个病的介绍，多半就是根据这篇论文写的。读到它，等于读到答案。

### 用了什么技术，为什么
规则：考试时，如果一个病在 HPO 里的资料**全部**来自这道题的论文（遮住以后一条都不剩），那么讲这个病的介绍也一起遮住。资料还有别的来源的病，介绍照常能搜到，和第 10 章"一条注释有别的出处就保留"是同一个标准。

dev-200 里，有 2 道题的正确答案会因此被遮住介绍。

### 公司里什么情况怎么做
- **资料不注明来源时，按"知识从哪来"推断**：不能因为没法精确遮就不遮
- **宁可遮多，不要漏**：泄露让分数虚高，比分数低一点糟糕得多

### 执行

In [ ]:
from raredx.data import cases as case_store
from raredx.evals import splits
from raredx.knowledge import masking

cases = {c.id: c for c in case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))}
dev = [cases[i] for i, _ in splits.read(ROOT / "evals/splits/dev-200.tsv")]
case = next(c for c in dev if c.id == 'PMID_27087320_Subject_2')
print("这道题：", case.disease_id, case.disease_label, " 出处", case.pmid)
mentions = [p for p in passages if case.disease_id in p.disease_ids]
print(f"讲这个病的介绍有 {len(mentions)} 块，例如 [{mentions[0].source}]：{mentions[0].text[:150]}……\n")
with masking.masked([case.pmid]):
    hidden = search.hpo.known_only_from(masking.current())
    found = search(mentions[0].text[:200], top_k=10)
print("遮住这篇论文以后，HPO 里只从它知道的病：", sorted(hidden))
print("用这段介绍原文去搜，能不能搜到它：", any(case.disease_id in h.disease_ids for h in found.hits))

### 结果怎么看
- 这个病在 HPO 里的资料全部来自这道题的论文，所以它的介绍也被遮住
- 哪怕直接拿介绍的原文去搜，也搜不到：遮挡在检索这一步就生效，agent 怎么问都拿不到

### 面试怎么说
> "The texts cite no papers, so they cannot be masked by reference. The rule follows the knowledge instead: while a case is evaluated, any passage about a disease HPO knows only from that case's paper is hidden — its description almost surely rests on the same paper."

---

## 第 5 步 · 单独评测检索

### 在干什么
不用模型，只看检索：把病人的症状写成一句话去搜，正确答案出现在前几名的比例（召回率）。三种方式都比一遍，再把混合检索做成基线 `disease-texts`，正式考一次。

### 背景
第 17 章说过：做 RAG 的公司都会单独量"检索找得准不准"。检索没找到，模型再聪明也答不对。三种方式选哪个，也要拿数据说话。

### 执行
在本机把 200 道题各搜三遍，大约一两分钟。

In [ ]:
import collections

mlflow.tracing.disable()          # 本机计算，不记追踪
hits = collections.Counter()
for c in dev:
    with masking.masked([c.pmid]):
        for mode in ("dense", "keyword", "hybrid"):
            credit = collections.defaultdict(float)      # 和基线同样的换算：一段讲几个病，就分几份
            for rank, p in enumerate(search.ranked("; ".join(c.hpo_labels), 50, mode)):
                for d in p.disease_ids:
                    credit[d] += 1 / (60 + rank) / len(p.disease_ids)
            diseases = sorted(credit, key=lambda d: (-credit[d], d))
            for k in (1, 10):
                hits[mode, k] += c.disease_id in diseases[:k]
mlflow.tracing.enable()
names = {"dense": "按意思（向量）", "keyword": "按关键词（BM25）", "hybrid": "混合"}
for mode, label in names.items():
    print(f"{label:<10} 第 1 名 {hits[mode, 1] / 2:.1f}%   前 10 名 {hits[mode, 10] / 2:.1f}%")

### 结果怎么看

| 只用文字检索，不用模型 | 第 1 名 | 前 10 名 |
|---|---|---|
| 按意思（向量） | 4.5% | 9.5% |
| 按关键词（BM25） | 6.0% | 15.5% |
| **混合** | **8.0%** | **17.0%** |

- **混合检索最好**：按意思找，能把同一类病都找出来；按关键词找，能抓住专有名词。两种合起来比单用任何一种都好
- **但三种都很低**，前 10 名最多 17%。疾病介绍讲的是"一类病"长什么样，亚型之间的文字几乎一样（比如几十种痉挛性截瘫）。另外，MedlinePlus 的一段介绍常常对应几十个 OMIM 亚型。换算成病的时候，按"一段讲几个病就分几份"分配功劳，否则一段话就会把前 10 名占满

再和前面的工具放在一起看（都不用模型，都在 dev-200 上，记在 MLflow 里）：

In [ ]:
from raredx.evals.harness import Harness

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
def latest(name, split="dev-200"):
    return next(r for r in harness.runs(split) if r["name"] == name and not r["code"].endswith("-dirty"))
for label, name in (("按病排名（第 10 章）", "phenotype-rank"), ("找相似病人（第 17 章）", "similar-patients"), ("查疾病介绍", "disease-texts")):
    m = harness.client.get_run(latest(name)["run_id"]).data.metrics
    print(f"{label:<14} 第 1 名 {m['disease_top1/mean']:.1%}   前 10 名 {m['disease_top10/mean']:.1%}")

### 结果怎么看
- **单独拿来找病，文字检索是三个里最弱的**：前 10 名 17.0%，另外两个都在 50% 上下
- 这在意料之中。它在这里的用处不是找病，而是让 agent 读候选病的介绍、在理由里写出处。值不值，第 20 章拿掉它考一次就知道
- 这也是一条很实际的经验：**RAG 不是万能的**。数据本身有结构的时候（症状编号、一个个病人），按结构查比按文字查准得多

### 公司里什么情况怎么做
- **检索方式用数据选**：同一批问题，三种方式各跑一遍，看召回率
- **别指望一种检索包打天下**：结构化的数据用结构去查（第 10、17 章），文字用文字检索；交给 agent 一起用

### 面试怎么说
> "Retrieval was measured before any model saw it: hybrid beat dense and BM25 alone, but text retrieval finds the exact disease far less often than the structured tools — 17% top-10 against about half — because descriptions differ little between subtypes. So here RAG supplies descriptions and citations, not candidates, and the ablation will say whether that is worth its tokens."

---

## 第 6 步 · 接进 agent 和 MCP，能拿掉

### 在干什么
把新工具交给 agent，也加进 MCP 服务；和第 17 章一样，能用 `--without search_disease_texts` 拿掉。

### 执行

In [ ]:
from raredx.agent import system_prompt

tools = ["rank_diseases", "check_match", "lookup_disease", "find_disease", "find_similar_patients", "search_disease_texts"]
print([line for line in system_prompt(tools).splitlines() if line.startswith("- search_disease_texts")][0], "\n")
smoke = next(r for r in harness.runs("ci-20") if r["name"] == "seeded-agent"
             and "search_disease_texts" in harness.client.get_run(r["run_id"]).data.params["tools"])
traces = harness.traces(smoke["run_id"])
used = sum(any(s.name == "search_disease_texts" for s in t.data.spans) for t in traces)
run_info = harness.client.get_run(smoke["run_id"]).data
m = run_info.metrics
print(f"ci-20 试跑：交了答案 {m['n_answered']:.0f}/20，其中 {used} 道用了 search_disease_texts")
example = next(t for t in traces if any(s.name == "search_disease_texts" for s in t.data.spans))
asked = [s.inputs.get("query") for s in example.data.spans if s.name == "search_disease_texts" and isinstance(s.inputs, dict) and s.inputs.get("query")]
asked = list(dict.fromkeys(asked))          # 每次调用记两层，去掉重复
print("它在一道题里搜过：", asked[:3])
print(f"泄露审计：{run_info.tags['leak_audit']}，读了 {m['leak/tool_outputs_read']:.0f} 次工具回答，"
      f"漏出本篇论文 {m['leak/own_paper_shown']:.0f} 次、只从本篇才知道的病 {m['leak/hidden_disease_shown']:.0f} 次")

### 结果怎么看
- 提示词里多了一行，告诉 agent 什么时候用：读候选病的介绍，或者找"其他工具不太看重的症状"对得上的病，并写明出处
- ci-20 试跑：20 道全部交了答案，每道都用了新工具。它会用自己的话写查询：有时把症状概括成一句话，有时直接搜一个它怀疑的病名（比如芳香族 L-氨基酸脱羧酶缺乏症）。泄露审计通过：读了 1,276 次工具回答，漏出 0 次
- 分数留到第 20 章大考

### 面试怎么说
> "The text search joins the agent as a removable tool, like the patient search: the ablation runs in chapter 20 will say what each is worth."

---

## 本章小结

| 做了什么 | 结果 |
|---|---|
| 资料 | Orphanet（CC BY 4.0）+ MedlinePlus（公有领域），登记版本和校验码 |
| 切块 | 7,790 块，每块带病名、OMIM 编号、出处 |
| 向量库 | FastEmbed 本机算向量，Qdrant 混合检索（向量 + BM25，RRF 合并） |
| 防泄露 | 只从本篇论文知道的病，介绍也遮住（dev-200 里 2 道题） |
| 单独评测 | 混合检索：第 1 名 8.0%，前 10 名 17.0%（不用模型） |
| 接进 agent | 能拿掉；ci-20 试跑通过 |

## 面试题

1. **Walk me through your RAG pipeline.**
   > "Two licensed sources, registered with checksums; chunked along the documents' own structure, each passage led by its disease's name; dense and BM25 vectors computed locally with FastEmbed; hybrid search in Qdrant with reciprocal-rank fusion; the agent calls it as a tool and cites the source."

2. **How do you evaluate retrieval separately from generation?**
   > "Recall of the confirmed disease among the retrieved passages, on the same 200 cases, for each retrieval mode — before any model sees the results. The hybrid won, so it is the default."

3. **How do you stop a RAG tool from leaking the test answer?**
   > "Free text cites no papers, so masking by reference is impossible. Any passage about a disease known only from the case's own paper is hidden — its description almost surely rests on that paper."

**下一章 19**：第 16 章计划里的基因工具。动手之前，有一个设计上的问题要先定下来（见对话）。